# Early Stopping Model Comparison

Compare models trained with early stopping (stratified 15% val, patience=3, min_delta=0.003).
Each model stops at its own best epoch — fair comparison across architectures.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import sys
sys.path.insert(0, '..')

from sparc.utils.metrics import c_index

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# === Run directories ===
# Two canonical runs reported in the paper.
# Adjust the relative paths if your checkpoints / fold results live elsewhere.
RUNS = {
    'Image Only':        'runs/mmp20_image_only_cos20_es_20260303_130951',
    'SPARC-Risk (k=64)': 'runs/mmp20_sq_cos20_knn64_es_20260303_132721',
}

# Cancer groups (18 groups: COAD + READ merged into CRC).
CANCER_GROUPS = {
    'BRCA': ['TCGA-BRCA'], 'UCEC': ['TCGA-UCEC'], 'KIRC': ['TCGA-KIRC'],
    'LGG':  ['TCGA-LGG'],  'LUAD': ['TCGA-LUAD'], 'HNSC': ['TCGA-HNSC'],
    'LUSC': ['TCGA-LUSC'], 'CRC':  ['TCGA-COAD', 'TCGA-READ'],
    'SKCM': ['TCGA-SKCM'], 'BLCA': ['TCGA-BLCA'], 'GBM':  ['TCGA-GBM'],
    'STAD': ['TCGA-STAD'], 'LIHC': ['TCGA-LIHC'], 'KIRP': ['TCGA-KIRP'],
    'CESC': ['TCGA-CESC'], 'SARC': ['TCGA-SARC'], 'PAAD': ['TCGA-PAAD'],
    'ESCA': ['TCGA-ESCA'],
}

In [ ]:
def load_run(run_dir):
    """Load all fold results and patient-level predictions."""
    run_dir = Path(run_dir)
    fold_files = sorted(run_dir.glob('fold_*_result.json'))
    fold_results = []
    all_patients = []
    for ff in fold_files:
        with open(ff) as f:
            data = json.load(f)
        fold_results.append(data)
        for p in data.get('test_patient_results', []):
            p['fold'] = data['fold']
            all_patients.append(p)
    return fold_results, pd.DataFrame(all_patients)

# Load all runs
data = {}
for name, path in RUNS.items():
    folds, patients = load_run(path)
    data[name] = {'folds': folds, 'patients': patients}
    epochs = [f.get('best_epoch', f.get('epoch', '?')) for f in folds]
    print(f'{name}: {len(folds)} folds, {len(patients)} patients, stopped at epochs {epochs}')

In [ ]:
# Compute per-cancer C-index from patient-level predictions for each fold
names = list(RUNS.keys())
EXCLUDE_FOLDS = []
use_folds = [f for f in range(5) if f not in EXCLUDE_FOLDS]

per_fold_per_group = {}  # name -> fold -> {group: c_index}
for name in names:
    per_fold_per_group[name] = {}
    patients = data[name]['patients']
    for fold in use_folds:
        fp = patients[patients['fold'] == fold]
        fold_groups = {}
        for group_name, members in CANCER_GROUPS.items():
            sub = fp[fp['cancer_type'].isin(members)]
            if len(sub) >= 5 and sub['event'].sum() >= 1:
                try:
                    ci = c_index(sub['time'].values, sub['event'].values, sub['risk'].values)
                    fold_groups[group_name] = ci
                except:
                    fold_groups[group_name] = np.nan
            else:
                fold_groups[group_name] = np.nan
        per_fold_per_group[name][fold] = fold_groups

# Also compute per-cancer C-index for ALL cancer types (not just CANCER_GROUPS)
per_fold_all_cancers = {}  # name -> fold -> {cancer_type: c_index}
for name in names:
    per_fold_all_cancers[name] = {}
    patients = data[name]['patients']
    for fold in use_folds:
        fp = patients[patients['fold'] == fold]
        fold_cancers = {}
        for ct in fp['cancer_type'].unique():
            sub = fp[fp['cancer_type'] == ct]
            if len(sub) >= 5 and sub['event'].sum() >= 1:
                try:
                    ci = c_index(sub['time'].values, sub['event'].values, sub['risk'].values)
                    fold_cancers[ct] = ci
                except:
                    fold_cancers[ct] = np.nan
            else:
                fold_cancers[ct] = np.nan
        per_fold_all_cancers[name][fold] = fold_cancers

# Per-group averages across folds
per_group_data = {}
for name in names:
    per_group_data[name] = {}
    for group_name in CANCER_GROUPS:
        fold_cs = np.array([per_fold_per_group[name][f].get(group_name, np.nan) for f in use_folds])
        n_valid = np.sum(~np.isnan(fold_cs))
        if n_valid >= 2:
            per_group_data[name][group_name] = {
                'c_index': float(np.nanmean(fold_cs)),
                'std': float(np.nanstd(fold_cs)),
                'n_folds': int(n_valid),
            }

# Macro averages per fold
fold_macro = {}  # name -> [fold0_macro, ...] -- ALL cancers in the data
fold_filtered = {}  # name -> [fold0_filtered, ...] -- only CANCER_GROUPS
for name in names:
    fold_macro[name] = []
    fold_filtered[name] = []
    for fold in use_folds:
        # Macro: all cancer types present in this fold (not just CANCER_GROUPS)
        all_cs = [v for v in per_fold_all_cancers[name][fold].values() if not np.isnan(v)]
        fold_macro[name].append(np.mean(all_cs) if all_cs else np.nan)
        # Filtered: only cancer groups defined in CANCER_GROUPS
        group_cs = [per_fold_per_group[name][fold].get(g, np.nan) for g in CANCER_GROUPS]
        group_cs = [v for v in group_cs if not np.isnan(v)]
        fold_filtered[name].append(np.mean(group_cs) if group_cs else np.nan)

# Store summary
for name in names:
    data[name]['fold_cs'] = fold_macro[name]
    data[name]['fold_filtered'] = fold_filtered[name]
    data[name]['macro_avg'] = np.nanmean(fold_macro[name])
    data[name]['filtered_macro'] = np.nanmean(fold_filtered[name])

# Display
print(f'{"Model":<20} {"Macro Avg C":>15} {"Filtered Macro C":>18} {"Best Epochs":>20}')
print('-' * 77)
for name in names:
    epochs = [f.get('best_epoch', '?') for f in data[name]['folds']]
    ep_str = ','.join(str(e) for e in epochs)
    print(f'{name:<20} {data[name]["macro_avg"]:>11.4f}\u00b1{np.std(fold_macro[name]):.3f} '
          f'{data[name]["filtered_macro"]:>14.4f}\u00b1{np.std(fold_filtered[name]):.3f} '
          f'{ep_str:>20}')


In [ ]:
# ── Paired bar chart: per-cancer C-index with bootstrap CIs + p-values ───────
# Per-fold evaluation (matching the summary table). Pure numpy for speed.
import multiprocessing as mp
from tqdm.auto import tqdm

baseline = 'Image Only'
sq_name = next(n for n in names if n != baseline)
N_BOOT = 1000

def fmt_p(p, n_boot):
    if p == 0:
        return f'p<{1/n_boot:.4f}'
    elif p < 0.001:
        return 'p<0.001'
    elif p < 0.01:
        return 'p<0.01'
    elif p < 0.05:
        return 'p<0.05'
    else:
        return f'p={p:.2f}'

patients_bl = data[baseline]['patients']
patients_sq = data[sq_name]['patients']

# Pre-merge and convert to numpy arrays for speed
_merged = patients_bl[['patient_id', 'fold', 'time', 'event', 'risk', 'cancer_type']].merge(
    patients_sq[['patient_id', 'fold', 'risk']],
    on=['patient_id', 'fold'], suffixes=('_bl', '_sq')
)

# Convert to numpy for fast indexing
_T = _merged['time'].values.astype(np.float64)
_E = _merged['event'].values.astype(np.float64)
_R_bl = _merged['risk_bl'].values.astype(np.float64)
_R_sq = _merged['risk_sq'].values.astype(np.float64)
_fold = _merged['fold'].values.astype(np.int32)
_ct = _merged['cancer_type'].values
_pid = _merged['patient_id'].values

# Pre-compute: for each unique patient, which rows belong to them
_unique_pids = np.unique(_pid)
_pid_to_rows = {pid: np.where(_pid == pid)[0] for pid in _unique_pids}
_folds = np.unique(_fold)

# Pre-compute: for each (fold, cancer_type), which row indices
_unique_cts = np.unique(_ct)
_ct_codes = np.array([np.searchsorted(_unique_cts, c) for c in _ct], dtype=np.int32)
_fold_ct_indices = {}
for f in _folds:
    for ci, ct in enumerate(_unique_cts):
        mask = (_fold == f) & (_ct_codes == ci)
        idx = np.where(mask)[0]
        if len(idx) >= 5:
            _fold_ct_indices[(f, ci)] = idx

def _perfold_cindex_np(row_indices):
    """Per-fold C-index using numpy arrays. Returns (c_bl, c_sq)."""
    from sparc.utils.metrics import c_index as _c_index
    # Build a mask of which rows are selected (for resampled data, rows may repeat)
    bl_cs, sq_cs = [], []
    for f in _folds:
        f_mask = _fold[row_indices] == f
        f_idx = row_indices[f_mask]
        if len(f_idx) < 5 or _E[f_idx].sum() < 1:
            continue
        try:
            bl_cs.append(_c_index(_T[f_idx], _E[f_idx], _R_bl[f_idx]))
            sq_cs.append(_c_index(_T[f_idx], _E[f_idx], _R_sq[f_idx]))
        except:
            continue
    if not bl_cs:
        return np.nan, np.nan
    return np.mean(bl_cs), np.mean(sq_cs)

def _bootstrap_group(args):
    """Bootstrap one cancer group or macro, using per-fold evaluation + numpy."""
    group_name, members, n_boot, is_macro = args
    from sparc.utils.metrics import c_index as _c_index
    rng = np.random.RandomState(42)

    if is_macro:
        # Filter to rows in CANCER_GROUPS only (not all cancer types)
        valid_cts = set()
        for mm in CANCER_GROUPS.values():
            valid_cts.update(mm)
        ct_mask = np.isin(_ct, list(valid_cts))
        all_rows = np.where(ct_mask)[0]
        pids_in_scope = np.unique(_pid[all_rows])
    else:
        ct_mask = np.isin(_ct, members)
        all_rows = np.where(ct_mask)[0]
        pids_in_scope = np.unique(_pid[all_rows])

    n = len(all_rows)
    if n < 10:
        return group_name, {}

    # Map pids to their row indices (within scope)
    pid_rows = {}
    for pid in pids_in_scope:
        pid_rows[pid] = np.intersect1d(_pid_to_rows[pid], all_rows)

    if is_macro:
        def _macro_calc(row_idx):
            fold_bl, fold_sq = [], []
            for f in _folds:
                f_mask = _fold[row_idx] == f
                f_rows = row_idx[f_mask]
                bl_cs, sq_cs = [], []
                for ci, ct in enumerate(_unique_cts):
                    ct_f_mask = _ct_codes[f_rows] == ci
                    cf_rows = f_rows[ct_f_mask]
                    if len(cf_rows) < 5 or _E[cf_rows].sum() < 1:
                        continue
                    try:
                        bl_cs.append(_c_index(_T[cf_rows], _E[cf_rows], _R_bl[cf_rows]))
                        sq_cs.append(_c_index(_T[cf_rows], _E[cf_rows], _R_sq[cf_rows]))
                    except:
                        continue
                if bl_cs:
                    fold_bl.append(np.mean(bl_cs))
                    fold_sq.append(np.mean(sq_cs))
            if not fold_bl:
                return np.nan, np.nan
            return np.mean(fold_bl), np.mean(fold_sq)

        c_bl, c_sq = _macro_calc(all_rows)
        boot_delta = np.empty(n_boot)
        pids_arr = np.array(list(pid_rows.keys()))
        n_pids = len(pids_arr)
        for b in range(n_boot):
            boot_pids = rng.choice(pids_arr, size=n_pids, replace=True)
            boot_rows = np.concatenate([pid_rows[p] for p in boot_pids])
            try:
                bb, bs = _macro_calc(boot_rows)
                boot_delta[b] = bs - bb
            except:
                boot_delta[b] = np.nan

        valid = boot_delta[~np.isnan(boot_delta)]
        return group_name, {
            'c_bl': c_bl, 'c_sq': c_sq, 'delta': c_sq - c_bl, 'n': n,
            'delta_lo': np.percentile(valid, 2.5), 'delta_hi': np.percentile(valid, 97.5),
            'p_value': np.mean(valid <= 0),
        }
    else:
        c_bl, c_sq = _perfold_cindex_np(all_rows)
        if np.isnan(c_bl):
            return group_name, {}

        pids_arr = np.array(list(pid_rows.keys()))
        n_pids = len(pids_arr)
        boot_bl = np.empty(n_boot)
        boot_sq = np.empty(n_boot)
        for b in range(n_boot):
            boot_pids = rng.choice(pids_arr, size=n_pids, replace=True)
            boot_rows = np.concatenate([pid_rows[p] for p in boot_pids])
            bb, bs = _perfold_cindex_np(boot_rows)
            boot_bl[b] = bb
            boot_sq[b] = bs

        valid = ~(np.isnan(boot_bl) | np.isnan(boot_sq))
        boot_delta = boot_sq[valid] - boot_bl[valid]
        return group_name, {
            'c_bl': c_bl, 'c_sq': c_sq, 'n': n,
            'bl_lo': np.percentile(boot_bl[valid], 2.5), 'bl_hi': np.percentile(boot_bl[valid], 97.5),
            'sq_lo': np.percentile(boot_sq[valid], 2.5), 'sq_hi': np.percentile(boot_sq[valid], 97.5),
            'delta': c_sq - c_bl,
            'delta_lo': np.percentile(boot_delta, 2.5), 'delta_hi': np.percentile(boot_delta, 97.5),
            'p_value': np.mean(boot_delta <= 0),
        }

# Build tasks (per-cancer only, macro = average)
tasks = [(g, m, N_BOOT, False) for g, m in CANCER_GROUPS.items()]

results = {}
with mp.Pool(18) as pool:
    for g, res in tqdm(pool.imap_unordered(_bootstrap_group, tasks), total=len(tasks), desc='Bootstrap'):
        if res:
            results[g] = res

# Macro = average of per-cancer C-indices
macro_res = {
    'c_bl': np.mean([results[g]['c_bl'] for g in results]),
    'c_sq': np.mean([results[g]['c_sq'] for g in results]),
    'delta': np.mean([results[g]['delta'] for g in results]),
    'n': sum(results[g]['n'] for g in results),
    
}

# Wilcoxon
per_cancer_deltas = [results[g]['delta'] for g in results]
_, wilcoxon_p = stats.wilcoxon(per_cancer_deltas, alternative='greater')

sorted_groups = sorted(results.keys())

In [ ]:
# ── Plot ──
fig, ax = plt.subplots(figsize=(max(12, (len(sorted_groups) + 1) * 0.7), 5.5))
x = np.arange(len(sorted_groups))
w = 0.40

bl_vals = [results[g]['c_bl'] for g in sorted_groups]
sq_vals = [results[g]['c_sq'] for g in sorted_groups]
bl_lo = [results[g]['c_bl'] - results[g]['bl_lo'] for g in sorted_groups]
bl_hi = [results[g]['bl_hi'] - results[g]['c_bl'] for g in sorted_groups]
sq_lo = [results[g]['c_sq'] - results[g]['sq_lo'] for g in sorted_groups]
sq_hi = [results[g]['sq_hi'] - results[g]['c_sq'] for g in sorted_groups]
ns = [results[g]['n'] for g in sorted_groups]

ax.bar(x - w/2, bl_vals, w, yerr=[bl_lo, bl_hi], capsize=2, error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
       color='#D1D1D1', edgecolor='black', linewidth=0.5, label='Image Only', zorder=2)
ax.bar(x + w/2, sq_vals, w, yerr=[sq_lo, sq_hi], capsize=2, error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
       color='#2B6B8A', edgecolor='black', linewidth=0.5, label='SPARC-Risk', zorder=2)





tick_labels = [f'{g}\n(n={ns[i]})' for i, g in enumerate(sorted_groups)]
ax.set_xticks(x)
ax.set_xticklabels(tick_labels, fontsize=8.5)
ax.set_ylabel('C-index (DSS)', fontsize=12)
ax.set_title('Per-Cancer C-index: SPARC-Risk vs Image Only', fontsize=13)
ax.legend(loc='upper left', fontsize=10)

n_pos = sum(1 for g in results if results[g]['delta'] > 0)
ax.text(0.98, 0.98,
        f'SQ higher in {n_pos}/{len(results)} cancers\n'
        f'Wilcoxon signed-rank {fmt_p(wilcoxon_p, 10000)}',
        transform=ax.transAxes, fontsize=9, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))

ax.set_ylim(0.52, 0.90)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figs/tcga_cindex_bootstrap.png', dpi=300, bbox_inches='tight')
plt.show()

# Print table
print(f'\n{"Cancer":>8s} {"n":>5s} {"IMG":>7s} {"SQ":>7s} {"Δ":>7s} {"95% CI":>18s} {"p(1s)":>10s}')
print('-' * 68)
for g in sorted_groups:
    r = results[g]
    sig = '***' if r['p_value'] < 0.001 else ('**' if r['p_value'] < 0.01 else ('*' if r['p_value'] < 0.05 else ''))
    print(f'{g:>8s} {r["n"]:>5d} {r["c_bl"]:>7.3f} {r["c_sq"]:>7.3f} {r["delta"]:>+7.3f} [{r["delta_lo"]:+.3f}, {r["delta_hi"]:+.3f}] {fmt_p(r["p_value"], N_BOOT):>10s} {sig}')
print(f'\nWilcoxon signed-rank p = {wilcoxon_p:.4f}')
print(f'* p<0.05  ** p<0.01  *** p<0.001 (one-sided paired bootstrap)')

## 6c. Paired Hazard Ratio Bar Chart with Bootstrap CIs

In [ ]:
# ── Paired HR bar chart with bootstrap CIs ───────────────────────────────────
# HR via median-split + log-rank (matching the original notebook approach).
import multiprocessing as mp
from tqdm.auto import tqdm

baseline = 'Image Only'
sq_name = next(n for n in names if n != baseline)
N_BOOT = 1000

def fmt_p(p, n_boot):
    if p == 0:
        return f'p<{1/n_boot:.4f}'
    elif p < 0.001:
        return 'p<0.001'
    elif p < 0.01:
        return 'p<0.01'
    elif p < 0.05:
        return 'p<0.05'
    else:
        return f'p={p:.2f}'

patients_bl = data[baseline]['patients']
patients_sq = data[sq_name]['patients']

_merged = patients_bl[['patient_id', 'fold', 'time', 'event', 'risk', 'cancer_type']].merge(
    patients_sq[['patient_id', 'fold', 'risk']],
    on=['patient_id', 'fold'], suffixes=('_bl', '_sq')
)

_T = _merged['time'].values.astype(np.float64) / 365.25  # convert to years
_E = _merged['event'].values.astype(np.float64)
_R_bl = _merged['risk_bl'].values.astype(np.float64)
_R_sq = _merged['risk_sq'].values.astype(np.float64)
_fold = _merged['fold'].values.astype(np.int32)
_ct = _merged['cancer_type'].values
_pid = _merged['patient_id'].values
_unique_pids = np.unique(_pid)
_pid_to_rows = {pid: np.where(_pid == pid)[0] for pid in _unique_pids}
_folds = np.unique(_fold)

def _logrank_hr(t, e, risk):
    """Median-split log-rank HR: high-risk vs low-risk."""
    med = np.median(risk)
    hi = risk >= med
    lo = ~hi
    if hi.sum() < 2 or lo.sum() < 2:
        return np.nan
    t_hi, e_hi = t[hi], e[hi]
    t_lo, e_lo = t[lo], e[lo]
    e_hi, e_lo = e_hi.astype(bool), e_lo.astype(bool)
    event_times = np.unique(np.concatenate([t_hi[e_hi], t_lo[e_lo]]))
    hr_num, hr_den = 0.0, 0.0
    for et in event_times:
        n1, n2 = np.sum(t_hi >= et), np.sum(t_lo >= et)
        n = n1 + n2
        d1 = np.sum((t_hi == et) & e_hi)
        d2 = np.sum((t_lo == et) & e_lo)
        if n < 2:
            continue
        hr_num += d1 * n2 / n
        hr_den += d2 * n1 / n
    return hr_num / hr_den if hr_den > 0 else np.inf

def _pooled_hr(row_indices):
    """Pooled HR across all folds."""
    if len(row_indices) < 10 or _E[row_indices].sum() < 2:
        return np.nan, np.nan
    hb = _logrank_hr(_T[row_indices], _E[row_indices], _R_bl[row_indices])
    hs = _logrank_hr(_T[row_indices], _E[row_indices], _R_sq[row_indices])
    if np.isfinite(hb) and np.isfinite(hs):
        return hb, hs
    return np.nan, np.nan

def _bootstrap_hr(args):
    """Bootstrap HR for one cancer group."""
    group_name, members, n_boot = args
    rng = np.random.RandomState(42)

    ct_mask = np.isin(_ct, members)
    all_rows = np.where(ct_mask)[0]
    pids_in_scope = np.unique(_pid[all_rows])
    n = len(all_rows)
    if n < 20:
        return group_name, {}

    pid_rows = {pid: np.intersect1d(_pid_to_rows[pid], all_rows) for pid in pids_in_scope}

    hr_bl, hr_sq = _pooled_hr(all_rows)
    if np.isnan(hr_bl):
        return group_name, {}

    pids_arr = np.array(list(pid_rows.keys()))
    boot_bl = np.empty(n_boot)
    boot_sq = np.empty(n_boot)
    for b in range(n_boot):
        boot_pids = rng.choice(pids_arr, size=len(pids_arr), replace=True)
        boot_rows = np.concatenate([pid_rows[p] for p in boot_pids])
        bb, bs = _pooled_hr(boot_rows)
        boot_bl[b] = bb
        boot_sq[b] = bs

    valid = ~(np.isnan(boot_bl) | np.isnan(boot_sq))
    boot_delta = boot_sq[valid] - boot_bl[valid]
    return group_name, {
        'hr_bl': hr_bl, 'hr_sq': hr_sq, 'n': n,
        'bl_lo': np.percentile(boot_bl[valid], 2.5), 'bl_hi': np.percentile(boot_bl[valid], 97.5),
        'sq_lo': np.percentile(boot_sq[valid], 2.5), 'sq_hi': np.percentile(boot_sq[valid], 97.5),
        'delta': hr_sq - hr_bl,
        'delta_lo': np.percentile(boot_delta, 2.5), 'delta_hi': np.percentile(boot_delta, 97.5),
        'p_value': np.mean(boot_delta <= 0),
    }

# Run per-cancer only
tasks = [(g, m, N_BOOT) for g, m in CANCER_GROUPS.items()]

hr_results = {}
with mp.Pool(18) as pool:
    for g, res in tqdm(pool.imap_unordered(_bootstrap_hr, tasks), total=len(tasks), desc='HR Bootstrap'):
        if res:
            hr_results[g] = res

# Wilcoxon on HR deltas
hr_deltas = [hr_results[g]['delta'] for g in hr_results]
_, hr_wilcoxon_p = stats.wilcoxon(hr_deltas, alternative='greater')

sorted_hr = sorted(hr_results.keys())

# Print table
print(f'\n{"Cancer":>8s} {"n":>5s} {"IMG HR":>7s} {"SQ HR":>7s} {"Δ":>7s} {"p(1s)":>10s}')
print('-' * 50)
for g in sorted_hr:
    r = hr_results[g]
    sig = '***' if r['p_value'] < 0.001 else ('**' if r['p_value'] < 0.01 else ('*' if r['p_value'] < 0.05 else ''))
    print(f'{g:>8s} {r["n"]:>5d} {r["hr_bl"]:>7.2f} {r["hr_sq"]:>7.2f} {r["delta"]:>+7.2f} {fmt_p(r["p_value"], N_BOOT):>10s} {sig}')
print(f'\nWilcoxon signed-rank p = {hr_wilcoxon_p:.4f}')

In [ ]:
# ── Plot (log scale) ──
fig, ax = plt.subplots(figsize=(max(12, (len(sorted_hr) + 1) * 0.7), 5.5))
x = np.arange(len(sorted_hr))
w = 0.40

bl_vals = np.array([hr_results[g]['hr_bl'] for g in sorted_hr])
sq_vals = np.array([hr_results[g]['hr_sq'] for g in sorted_hr])
bl_lo_vals = np.array([hr_results[g]['bl_lo'] for g in sorted_hr])
bl_hi_vals = np.array([hr_results[g]['bl_hi'] for g in sorted_hr])
sq_lo_vals = np.array([hr_results[g]['sq_lo'] for g in sorted_hr])
sq_hi_vals = np.array([hr_results[g]['sq_hi'] for g in sorted_hr])
ns = [hr_results[g]['n'] for g in sorted_hr]

# Clip CIs to a reasonable range for display

# For log-scale bars, we draw bars starting from 1 (HR=1 is the null)
# bar height = HR value, plotted on log scale
ax.set_yscale('log')

ax.bar(x - w/2, bl_vals, w,
       yerr=[bl_vals - bl_lo_vals, bl_hi_vals - bl_vals], capsize=2, error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
       color='#D1D1D1', edgecolor='black', linewidth=0.5, label='Image Only', zorder=2)
ax.bar(x + w/2, sq_vals, w,
       yerr=[sq_vals - sq_lo_vals, sq_hi_vals - sq_vals], capsize=2, error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
       color='#2B6B8A', edgecolor='black', linewidth=0.5, label='SPARC-Risk', zorder=2)

ax.axhline(y=1.0, color='black', linewidth=1, linestyle='--', alpha=0.5)

tick_labels = [f'{g}\n(n={ns[i]})' for i, g in enumerate(sorted_hr)]
ax.set_xticks(x)
ax.set_xticklabels(tick_labels, fontsize=8.5)
ax.set_ylabel('Hazard Ratio (log scale)', fontsize=12)
ax.set_title('Per-Cancer Hazard Ratio: SPARC-Risk vs Image Only', fontsize=13)
ax.legend(loc='upper left', fontsize=10)

n_pos = sum(1 for g in hr_results if hr_results[g]['delta'] > 0)
ax.text(0.98, 0.98,
        f'SQ HR higher in {n_pos}/{len(hr_results)} cancers\n'
        f'Wilcoxon signed-rank {fmt_p(hr_wilcoxon_p, 10000)}',
        transform=ax.transAxes, fontsize=9, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))

ax.set_ylim(0.8, None)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}'))
ax.grid(axis='y', alpha=0.3, which='both')
plt.ylim((0, 14))
plt.tight_layout()
plt.savefig('../figs/tcga_hr_bootstrap.png', dpi=300, bbox_inches='tight')
plt.show()

# Print table
print(f'\n{"Cancer":>8s} {"n":>5s} {"IMG HR":>7s} {"SQ HR":>7s} {"Δ":>7s} {"p(1s)":>10s}')
print('-' * 50)
for g in sorted_hr:
    r = hr_results[g]
    print(f'{g:>8s} {r["n"]:>5d} {r["hr_bl"]:>7.2f} {r["hr_sq"]:>7.2f} {r["delta"]:>+7.2f} {fmt_p(r["p_value"], N_BOOT):>10s}')
print(f'\nWilcoxon signed-rank p = {hr_wilcoxon_p:.4f}')


In [ ]:
def kaplan_meier(times, events):
    order = np.argsort(times)
    t = times[order]
    e = events[order].astype(bool)
    event_times = np.unique(t[e])
    km_t, km_s, km_lo, km_hi = [0.0], [1.0], [1.0], [1.0]
    s, gw = 1.0, 0.0
    for et in event_times:
        n_i = np.sum(t >= et)
        d_i = np.sum((t == et) & e)
        if n_i == 0: break
        s *= (1 - d_i / n_i)
        if n_i > d_i: gw += d_i / (n_i * (n_i - d_i))
        se = s * np.sqrt(gw)
        km_t.append(et); km_s.append(s)
        km_lo.append(max(0, s - 1.96 * se)); km_hi.append(min(1, s + 1.96 * se))
    return np.array(km_t), np.array(km_s), np.array(km_lo), np.array(km_hi)

def logrank_p(t1, e1, t2, e2):
    e1, e2 = e1.astype(bool), e2.astype(bool)
    event_times = np.unique(np.concatenate([t1[e1], t2[e2]]))
    O1, E1, V = 0.0, 0.0, 0.0
    hr_num, hr_den = 0.0, 0.0
    for et in event_times:
        n1, n2 = np.sum(t1 >= et), np.sum(t2 >= et)
        n = n1 + n2
        d1 = np.sum((t1 == et) & e1)
        d2 = np.sum((t2 == et) & e2)
        d = d1 + d2
        if n < 2: continue
        O1 += d1; E1 += d * n1 / n
        V += d * n1 * n2 * (n - d) / (n**2 * (n - 1))
        hr_num += d1 * n2 / n; hr_den += d2 * n1 / n
    if V < 1e-10: return 1.0, 1.0
    p = float(stats.chi2.sf((O1 - E1)**2 / V, df=1))
    hr = hr_num / hr_den if hr_den > 0 else np.inf
    return p, hr

# Plot KM curves per cancer for each model
cancer_names = list(CANCER_GROUPS.keys())
n_cancers = len(cancer_names)
n_cols = 5
n_rows = (n_cancers + n_cols - 1) // n_cols

for model_name in data.keys():
    patients = data[model_name]['patients']
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 4 * n_rows))
    axes_flat = axes.flatten()
    for i, cname in enumerate(cancer_names):
        ax = axes_flat[i]
        members = CANCER_GROUPS[cname]
        sub = patients[patients['cancer_type'].isin(members)]
        if len(sub) < 5 or sub['event'].sum() < 1:
            ax.text(0.5, 0.5, f'{cname}\nInsufficient data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=10)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            continue
        t = sub['time'].values / 365.25
        e = sub['event'].values
        r = sub['risk'].values
        med = np.median(r)
        hi_mask = r >= med; lo_mask = ~hi_mask
        if hi_mask.sum() < 2 or lo_mask.sum() < 2:
            ax.text(0.5, 0.5, f'{cname}\nCannot split', transform=ax.transAxes,
                    ha='center', va='center', fontsize=10)
            continue
        t_hi, s_hi, lo_hi, hi_hi = kaplan_meier(t[hi_mask], e[hi_mask])
        t_lo, s_lo, lo_lo, hi_lo = kaplan_meier(t[lo_mask], e[lo_mask])
        p, hr = logrank_p(t[hi_mask], e[hi_mask], t[lo_mask], e[lo_mask])
        ax.step(t_hi, s_hi, where='post', color='#D32F2F', linewidth=1.5, label='High risk')
        ax.fill_between(t_hi, lo_hi, hi_hi, step='post', alpha=0.15, color='#D32F2F')
        ax.step(t_lo, s_lo, where='post', color='#1976D2', linewidth=1.5, label='Low risk')
        ax.fill_between(t_lo, lo_lo, hi_lo, step='post', alpha=0.15, color='#1976D2')
        p_str = f'p={p:.1e}' if p < 0.01 else f'p={p:.3f}'
        ax.set_title(f'{cname} (n={len(sub)})\nHR={hr:.2f}, {p_str}', fontsize=9)
        ax.set_xlim(0, min(10, t.max() * 1.05))
        ax.set_ylim(0, 1.05)
        if i == 0: ax.legend(fontsize=7, loc='lower left')
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')
    fig.suptitle(f'{model_name) — KM Curves', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f'../figs/es_km_{model_name.lower().replace(" ", "_")}.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'../figs/es_km_{model_name.lower().replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Hazard Ratio Comparison

## 11. Per-Cancer Fusion Gate Analysis (2×2)

The learned fusion gate controls how much biological enrichment (from molecular programs)
is added to the image representation.

- **(a) Log-odds** — raw logit before sigmoid. 0 = neutral, positive = more biology.
- **(b) Gate vs ΔC-index** — correlation with bootstrap slope CI.
- **(c) LOO strip plot** — drop each cancer; shows no single point drives the result.
- **(d) Gate vs Δlog(HR)** — alternative benefit metric confirms the trend.

In [ ]:
next((n for n in names if 'SPARC-Risk' in n), None)

In [ ]:
import torch
sys.path.insert(0, '..')
from sparc.data.dataset import CANCER_TYPE_ORDER, CANCER_TYPE_TO_IDX
from scipy.stats import pearsonr, spearmanr

# Load gate values from checkpoints — use whichever SPARC-Risk run is in RUNS
SPARC_KEY = 'SPARC-Risk (k=64)'
SPARC_RUN = RUNS[SPARC_KEY] if SPARC_KEY else list(RUNS.values())[-1]

CANCERS_19 = ['TCGA-BRCA', 'TCGA-UCEC', 'TCGA-KIRC', 'TCGA-LGG', 'TCGA-LUAD',
              'TCGA-HNSC', 'TCGA-LUSC', 'TCGA-COAD', 'TCGA-READ', 'TCGA-SKCM',
              'TCGA-BLCA', 'TCGA-GBM', 'TCGA-STAD', 'TCGA-LIHC', 'TCGA-KIRP',
              'TCGA-CESC', 'TCGA-SARC', 'TCGA-PAAD', 'TCGA-ESCA']

gate_logits = {ct: [] for ct in CANCERS_19}
for fold in range(5):
    ckpt = torch.load(f'{SPARC_RUN}/checkpoints/fold_{fold}_best.pt', map_location='cpu')
    gate = ckpt['model_state_dict']['fusion.fusion_gate']
    for ct in CANCERS_19:
        idx = CANCER_TYPE_TO_IDX[ct]
        gate_logits[ct].append(gate[idx].item())

# Aggregate gate logits to evaluation groups (COAD+READ -> CRC)
group_gate = {}
group_gate_std = {}
for group_name, members in CANCER_GROUPS.items():
    all_fold_means = []
    for fold in range(5):
        fold_vals = []
        for ct in members:
            if ct in gate_logits:
                fold_vals.append(gate_logits[ct][fold])
        if fold_vals:
            all_fold_means.append(np.mean(fold_vals))
    if all_fold_means:
        group_gate[group_name] = np.mean(all_fold_means)
        group_gate_std[group_name] = np.std(all_fold_means)

# Sort groups by gate logit for panel (a)
group_names_sorted = sorted(group_gate.keys(), key=lambda g: group_gate[g], reverse=True)
group_logits_sorted = np.array([group_gate[g] for g in group_names_sorted])
group_stds_sorted = np.array([group_gate_std[g] for g in group_names_sorted])

# Build scatter data at group level
delta_c = []
gate_vals = []
labels_scatter = []
for group_name in CANCER_GROUPS:
    if group_name not in group_gate: continue
    if group_name not in per_group_data.get(SPARC_KEY, {}): continue
    if group_name not in per_group_data.get('Image Only', {}): continue
    dc = per_group_data[SPARC_KEY][group_name]['c_index'] - per_group_data['Image Only'][group_name]['c_index']
    delta_c.append(dc)
    gate_vals.append(group_gate[group_name])
    labels_scatter.append(group_name)

gate_vals = np.array(gate_vals)
delta_c = np.array(delta_c)
n_pts = len(gate_vals)

# Precompute LOO
loo_pearson, loo_spearman, dropped_labels = [], [], []
for i in range(n_pts):
    mask = np.ones(n_pts, dtype=bool); mask[i] = False
    r_p_i, _ = pearsonr(gate_vals[mask], delta_c[mask])
    r_s_i, _ = spearmanr(gate_vals[mask], delta_c[mask])
    loo_pearson.append(r_p_i); loo_spearman.append(r_s_i)
    dropped_labels.append(labels_scatter[i])
loo_pearson = np.array(loo_pearson); loo_spearman = np.array(loo_spearman)
r_p_full, p_p_full = pearsonr(gate_vals, delta_c)
r_s_full, p_s_full = spearmanr(gate_vals, delta_c)

# Precompute delta log(HR)
delta_log_hr, gate_vals_hr, labels_hr = [], [], []
for group_name in labels_scatter:
    if group_name not in hr_data.get(SPARC_KEY, {}) or group_name not in hr_data.get('Image Only', {}):
        continue
    hr_sparc = hr_data[SPARC_KEY][group_name]['hr']
    hr_img = hr_data['Image Only'][group_name]['hr']
    if not np.isfinite(hr_sparc) or not np.isfinite(hr_img) or hr_sparc <= 0 or hr_img <= 0:
        continue
    delta_log_hr.append(np.log(hr_sparc) - np.log(hr_img))
    gate_vals_hr.append(gate_vals[labels_scatter.index(group_name)])
    labels_hr.append(group_name)
gate_vals_hr = np.array(gate_vals_hr); delta_log_hr = np.array(delta_log_hr)

# ============================================================
# 2x2 Figure
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 13))

# --- (a) Log-odds bar chart (CRC grouped) ---
ax = axes[0, 0]
colors_gate = ['#2B6B8A' if v > 0 else '#D32F2F' for v in group_logits_sorted]
ax.barh(range(len(group_names_sorted)), group_logits_sorted,
        xerr=group_stds_sorted, capsize=3,
        color=colors_gate, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(group_names_sorted)))
ax.set_yticklabels(group_names_sorted, fontsize=9)
ax.axvline(x=0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Gate log-odds (logit)', fontsize=10)
ax.invert_yaxis()

# --- (b) Gate vs delta C-index ---
ax = axes[0, 1]
ax.scatter(gate_vals, delta_c, s=80, c='#2B6B8A', edgecolors='black', linewidth=0.7, zorder=3)
for x_pt, y_pt, label in zip(gate_vals, delta_c, labels_scatter):
    ax.annotate(label, (x_pt, y_pt), textcoords='offset points', xytext=(6, 6), fontsize=7.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Gate log-odds (logit)', fontsize=10)
ax.set_ylabel(f'\u0394C-index ({SPARC_KEY} \u2212 Image Only)', fontsize=10)

# Bootstrap slope CI
n_boot = 5000; boot_rng = np.random.RandomState(42)
boot_slopes, boot_intercepts = [], []
for _ in range(n_boot):
    idx = boot_rng.choice(n_pts, n_pts, replace=True)
    bz = np.polyfit(gate_vals[idx], delta_c[idx], 1)
    boot_slopes.append(bz[0]); boot_intercepts.append(bz[1])
boot_slopes = np.array(boot_slopes); boot_intercepts = np.array(boot_intercepts)
slope_lo, slope_hi = np.percentile(boot_slopes, [2.5, 97.5])
slope_point = np.polyfit(gate_vals, delta_c, 1)
x_line = np.linspace(gate_vals.min() - 0.02, gate_vals.max() + 0.02, 100)
ax.plot(x_line, np.polyval(slope_point, x_line), '--', color='#D32F2F', alpha=0.6, linewidth=1.5, zorder=2)
y_boot = np.array([s * x_line + b for s, b in zip(boot_slopes, boot_intercepts)])
ax.fill_between(x_line, np.percentile(y_boot, 2.5, axis=0), np.percentile(y_boot, 97.5, axis=0),
                alpha=0.12, color='#D32F2F', zorder=1)
ax.text(0.05, 0.95,
        f'n={n_pts} cancer groups\n'
        f'Pearson r={r_p_full:.3f} (p={p_p_full:.3f})\n'
        f'Spearman \u03c1={r_s_full:.3f} (p={p_s_full:.3f})\n'
        f'Slope={slope_point[0]:.3f} [{slope_lo:.3f}, {slope_hi:.3f}]',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))

# --- (c) LOO strip plot ---
ax = axes[1, 0]
sort_loo = np.argsort(loo_spearman)
sorted_labels = np.array(dropped_labels)[sort_loo]
sorted_spearman = loo_spearman[sort_loo]
sorted_pearson = loo_pearson[sort_loo]
y_pos = np.arange(n_pts)
ax.scatter(sorted_spearman, y_pos, s=60, c='#2B6B8A', edgecolors='black',
           linewidth=0.5, zorder=3, label='Spearman \u03c1')
ax.scatter(sorted_pearson, y_pos, s=60, c='#E8A855', edgecolors='black',
           linewidth=0.5, zorder=3, marker='D', label='Pearson r')
for y, sp, pe in zip(y_pos, sorted_spearman, sorted_pearson):
    ax.plot([sp, pe], [y, y], color='gray', linewidth=0.5, alpha=0.5, zorder=1)
ax.axvline(x=r_s_full, color='#2B6B8A', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Full \u03c1={r_s_full:.3f}')
ax.axvline(x=r_p_full, color='#E8A855', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Full r={r_p_full:.3f}')
ax.axvline(x=0, color='black', linewidth=0.8, alpha=0.3)
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_labels, fontsize=9)
ax.set_xlabel('Correlation after dropping cancer', fontsize=10)
ax.legend(fontsize=7.5, loc='lower right', framealpha=0.9)
ax.invert_yaxis()
all_positive = np.all(loo_spearman > 0) and np.all(loo_pearson > 0)
ax.text(0.02, 0.02,
        f'Range \u03c1: [{loo_spearman.min():.3f}, {loo_spearman.max():.3f}]\n'
        f'Range r: [{loo_pearson.min():.3f}, {loo_pearson.max():.3f}]\n'
        f'All positive: {all_positive}',
        transform=ax.transAxes, fontsize=8, va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))

# --- (d) Gate vs delta log(HR) ---
ax = axes[1, 1]
ax.scatter(gate_vals_hr, delta_log_hr, s=80, c='#2B6B8A', edgecolors='black', linewidth=0.7, zorder=3)
for x_pt, y_pt, label in zip(gate_vals_hr, delta_log_hr, labels_hr):
    ax.annotate(label, (x_pt, y_pt), textcoords='offset points', xytext=(6, 6), fontsize=7.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Gate log-odds (logit)', fontsize=10)
ax.set_ylabel(f'\u0394log(HR) ({SPARC_KEY} \u2212 Image Only)', fontsize=10)

if len(gate_vals_hr) > 3:
    r_p_hr, p_p_hr = pearsonr(gate_vals_hr, delta_log_hr)
    r_s_hr, p_s_hr = spearmanr(gate_vals_hr, delta_log_hr)
    z_hr = np.polyfit(gate_vals_hr, delta_log_hr, 1)
    x_line_hr = np.linspace(gate_vals_hr.min() - 0.02, gate_vals_hr.max() + 0.02, 100)
    ax.plot(x_line_hr, np.polyval(z_hr, x_line_hr), '--', color='#D32F2F', alpha=0.6, linewidth=1.5, zorder=2)
    boot_rng_hr = np.random.RandomState(42)
    boot_slopes_hr, boot_intercepts_hr = [], []
    for _ in range(5000):
        idx = boot_rng_hr.choice(len(gate_vals_hr), len(gate_vals_hr), replace=True)
        bz = np.polyfit(gate_vals_hr[idx], delta_log_hr[idx], 1)
        boot_slopes_hr.append(bz[0]); boot_intercepts_hr.append(bz[1])
    boot_slopes_hr = np.array(boot_slopes_hr); boot_intercepts_hr = np.array(boot_intercepts_hr)
    slope_lo_hr, slope_hi_hr = np.percentile(boot_slopes_hr, [2.5, 97.5])
    y_boot_hr = np.array([s * x_line_hr + b for s, b in zip(boot_slopes_hr, boot_intercepts_hr)])
    ax.fill_between(x_line_hr, np.percentile(y_boot_hr, 2.5, axis=0),
                    np.percentile(y_boot_hr, 97.5, axis=0), alpha=0.12, color='#D32F2F', zorder=1)
    ax.text(0.05, 0.95,
            f'n={len(gate_vals_hr)} cancer groups\n'
            f'Pearson r={r_p_hr:.3f} (p={p_p_hr:.3f})\n'
            f'Spearman \u03c1={r_s_hr:.3f} (p={p_s_hr:.3f})\n'
            f'Slope={z_hr[0]:.3f} [{slope_lo_hr:.3f}, {slope_hi_hr:.3f}]',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.savefig('../figs/gate_analysis_2x2.pdf', dpi=150, bbox_inches='tight')
plt.savefig('../figs/gate_analysis_2x2.png', dpi=300, bbox_inches='tight')
plt.show()

# Also save individual panels
for suffix in ['pdf', 'png']:
    dpi = 150 if suffix == 'pdf' else 300
    # (a)
    fig_a, ax_a = plt.subplots(figsize=(5, 7))
    colors_gate = ['#2B6B8A' if v > 0 else '#D32F2F' for v in group_logits_sorted]
    ax_a.barh(range(len(group_names_sorted)), group_logits_sorted, xerr=group_stds_sorted, capsize=3,
              color=colors_gate, edgecolor='black', linewidth=0.5)
    ax_a.set_yticks(range(len(group_names_sorted))); ax_a.set_yticklabels(group_names_sorted, fontsize=10)
    ax_a.axvline(x=0, color='black', linewidth=1, linestyle='--')
    ax_a.set_xlabel('Gate log-odds (logit)', fontsize=11)
    ax_a.invert_yaxis(); plt.tight_layout()
    fig_a.savefig(f'../figs/gate_a_logodds.{suffix}', dpi=dpi, bbox_inches='tight'); plt.close(fig_a)
    # (b)
    fig_b, ax_b = plt.subplots(figsize=(6, 5.5))
    ax_b.scatter(gate_vals, delta_c, s=90, c='#2B6B8A', edgecolors='black', linewidth=0.7, zorder=3)
    for x_pt, y_pt, label in zip(gate_vals, delta_c, labels_scatter):
        ax_b.annotate(label, (x_pt, y_pt), textcoords='offset points', xytext=(6, 6), fontsize=8.5)
    ax_b.axhline(y=0, color='gray', linestyle='--', alpha=0.5); ax_b.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax_b.set_xlabel('Gate log-odds (logit)', fontsize=11); ax_b.set_ylabel(f'\u0394C-index ({SPARC_KEY} \u2212 Image Only)', fontsize=11)
    ax_b.plot(x_line, np.polyval(slope_point, x_line), '--', color='#D32F2F', alpha=0.6, linewidth=1.5, zorder=2)
    ax_b.fill_between(x_line, np.percentile(y_boot, 2.5, axis=0), np.percentile(y_boot, 97.5, axis=0),
                      alpha=0.12, color='#D32F2F', zorder=1)
    ax_b.text(0.05, 0.95, f'n={n_pts} cancer groups\nPearson r={r_p_full:.3f} (p={p_p_full:.3f})\nSpearman \u03c1={r_s_full:.3f} (p={p_s_full:.3f})\nSlope={slope_point[0]:.3f} [{slope_lo:.3f}, {slope_hi:.3f}]',
              transform=ax_b.transAxes, fontsize=8.5, va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))
    plt.tight_layout(); fig_b.savefig(f'../figs/gate_b_delta_cindex.{suffix}', dpi=dpi, bbox_inches='tight'); plt.close(fig_b)
    # (c)
    fig_c, ax_c = plt.subplots(figsize=(6, 7))
    ax_c.scatter(sorted_spearman, y_pos, s=70, c='#2B6B8A', edgecolors='black', linewidth=0.5, zorder=3, label='Spearman \u03c1')
    ax_c.scatter(sorted_pearson, y_pos, s=70, c='#E8A855', edgecolors='black', linewidth=0.5, zorder=3, marker='D', label='Pearson r')
    for y, sp, pe in zip(y_pos, sorted_spearman, sorted_pearson):
        ax_c.plot([sp, pe], [y, y], color='gray', linewidth=0.5, alpha=0.5, zorder=1)
    ax_c.axvline(x=r_s_full, color='#2B6B8A', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Full \u03c1={r_s_full:.3f}')
    ax_c.axvline(x=r_p_full, color='#E8A855', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Full r={r_p_full:.3f}')
    ax_c.axvline(x=0, color='black', linewidth=0.8, alpha=0.3)
    ax_c.set_yticks(y_pos); ax_c.set_yticklabels(sorted_labels, fontsize=9.5)
    ax_c.set_xlabel('Correlation after dropping cancer', fontsize=11)
    ax_c.legend(fontsize=8.5, loc='lower right', framealpha=0.9); ax_c.invert_yaxis()
    ax_c.text(0.02, 0.02, f'Range \u03c1: [{loo_spearman.min():.3f}, {loo_spearman.max():.3f}]\nRange r: [{loo_pearson.min():.3f}, {loo_pearson.max():.3f}]\nAll positive: {all_positive}',
              transform=ax_c.transAxes, fontsize=8.5, va='bottom', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))
    plt.tight_layout(); fig_c.savefig(f'../figs/gate_c_loo.{suffix}', dpi=dpi, bbox_inches='tight'); plt.close(fig_c)
    # (d)
    fig_d, ax_d = plt.subplots(figsize=(6, 5.5))
    ax_d.scatter(gate_vals_hr, delta_log_hr, s=90, c='#2B6B8A', edgecolors='black', linewidth=0.7, zorder=3)
    for x_pt, y_pt, label in zip(gate_vals_hr, delta_log_hr, labels_hr):
        ax_d.annotate(label, (x_pt, y_pt), textcoords='offset points', xytext=(6, 6), fontsize=8)
    ax_d.axhline(y=0, color='gray', linestyle='--', alpha=0.5); ax_d.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax_d.set_xlabel('Gate log-odds (logit)', fontsize=11); ax_d.set_ylabel(f'\u0394log(HR) ({SPARC_KEY} \u2212 Image Only)', fontsize=11)
    if len(gate_vals_hr) > 3:
        ax_d.plot(x_line_hr, np.polyval(z_hr, x_line_hr), '--', color='#D32F2F', alpha=0.6, linewidth=1.5, zorder=2)
        ax_d.fill_between(x_line_hr, np.percentile(y_boot_hr, 2.5, axis=0), np.percentile(y_boot_hr, 97.5, axis=0),
                          alpha=0.12, color='#D32F2F', zorder=1)
        ax_d.text(0.05, 0.95, f'n={len(gate_vals_hr)} cancer groups\nPearson r={r_p_hr:.3f} (p={p_p_hr:.3f})\nSpearman \u03c1={r_s_hr:.3f} (p={p_s_hr:.3f})\nSlope={z_hr[0]:.3f} [{slope_lo_hr:.3f}, {slope_hi_hr:.3f}]',
                  transform=ax_d.transAxes, fontsize=8.5, va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9))
    plt.tight_layout(); fig_d.savefig(f'../figs/gate_d_delta_loghr.{suffix}', dpi=dpi, bbox_inches='tight'); plt.close(fig_d)

print('Saved: gate_analysis_2x2.{pdf,png} + 4 individual panels (a-d) as pdf+png')
print(f'\nLOO Spearman range: [{loo_spearman.min():.3f}, {loo_spearman.max():.3f}], all positive: {np.all(loo_spearman > 0)}')
print(f'LOO Pearson range:  [{loo_pearson.min():.3f}, {loo_pearson.max():.3f}], all positive: {np.all(loo_pearson > 0)}')
print(f'Most influential: dropping {dropped_labels[np.argmin(loo_spearman)]} \u2192 \u03c1={loo_spearman.min():.3f}')

# Print gate table
print(f'\n{"Group":<8} {"Logit":>8} {"Sigmoid":>8}')
print('-' * 28)
for g in group_names_sorted:
    sig = 1 / (1 + np.exp(-group_gate[g]))
    print(f'{g:<8} {group_gate[g]:>8.4f} {sig:>8.4f}')